In [0]:
%sql

CREATE OR REPLACE TABLE commodity_successful AS

SELECT
    third_party_id,
    groww_order_id,
    contract_id,
    gui_order_id,
    exchange_order_id,
    created_at,
    trading_symbol,
    symbol_isin,
    qty,
    filled_qty,
    avg_fill_price,
    order_type,
    buy_sell,
    underlying,
    future_option,
    call_put,
    entry_exit_flag,
    platform,
    exchange,
    contract_expiry_date,

    20 AS commodity_revenue

FROM commodity_orders

WHERE UPPER(TRIM(order_status)) = 'EXECUTED';

In [0]:
%sql

CREATE OR REPLACE TABLE commodity_user_metrics AS

SELECT

    third_party_id,

    -- Revenue
    COUNT(DISTINCT groww_order_id) * 20
        AS total_brokerage,

    -- Trading activity
    COUNT(DISTINCT groww_order_id)
        AS total_executed_orders,

    COUNT(DISTINCT TO_DATE(created_at))
        AS active_trading_days,

    -- Contract diversification
    COUNT(DISTINCT contract_id)
        AS distinct_contracts,

    COUNT(DISTINCT trading_symbol)
        AS distinct_symbols,

    COUNT(DISTINCT underlying)
        AS distinct_underlyings,

    -- Trading preferences
    COUNT(DISTINCT future_option)
        AS distinct_instrument_types,

    SUM(CASE
        WHEN future_option = 'FUT'
        THEN 1 ELSE 0
    END) AS futures_orders,

    SUM(CASE
        WHEN future_option = 'OPT'
        THEN 1 ELSE 0
    END) AS options_orders,

    SUM(CASE
        WHEN buy_sell = 'B'
        THEN 1 ELSE 0
    END) AS buy_orders,

    SUM(CASE
        WHEN buy_sell = 'S'
        THEN 1 ELSE 0
    END) AS sell_orders,

    MAX(created_at)
        AS last_order_date

FROM commodity_successful

GROUP BY third_party_id;

In [0]:
%sql

CREATE OR REPLACE TABLE commodity_user_segments AS

SELECT
    *,
    NTILE(5) OVER (
        ORDER BY total_brokerage DESC
    ) AS value_segment

FROM commodity_user_metrics;

In [0]:
%sql

SELECT

    value_segment,

    COUNT(*) AS total_users,

    SUM(total_executed_orders) AS total_orders,

    SUM(total_brokerage) AS total_revenue,

    ROUND(
        100.0 * SUM(total_brokerage) /
        NULLIF(SUM(SUM(total_brokerage)) OVER (), 0),
        2
    ) AS revenue_contribution_pct

FROM commodity_user_segments

GROUP BY value_segment

ORDER BY value_segment;

In [0]:
%sql

SELECT

    CASE
        WHEN value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    COUNT(*) AS total_users,

    ROUND(AVG(total_brokerage), 2)
        AS avg_revenue,

    ROUND(AVG(total_executed_orders), 2)
        AS avg_orders,

    ROUND(AVG(active_trading_days), 2)
        AS avg_active_days,

    ROUND(AVG(distinct_contracts), 2)
        AS avg_contracts,

    ROUND(AVG(distinct_underlyings), 2)
        AS avg_underlyings,

    ROUND(AVG(distinct_symbols), 2)
        AS avg_symbols

FROM commodity_user_segments

GROUP BY 1;

Analyse Futures vs Options preferences
We should understand which instrument types are more commonly used by high-value users.

In [0]:
%sql

SELECT

    CASE
        WHEN s.value_segment = 1 THEN 'High Value'
        ELSE 'Other Users'
    END AS user_category,

    o.future_option,

    COUNT(DISTINCT o.third_party_id) AS users,

    COUNT(DISTINCT o.groww_order_id) AS executed_orders,

    SUM(o.commodity_revenue) AS revenue

FROM commodity_successful o

JOIN commodity_user_segments s
    ON o.third_party_id = s.third_party_id

GROUP BY 1, 2

ORDER BY 1, 2;